# Étape 1 — Acquisition & Ingestion Lakehouse Bronze (HDFS 3.5 + Delta Lake)

> **Référentiel méthodologique :** [Introduction à la Data Science — Mission *Data Detective* (`modules.apti.space`)](https://modules.apti.space/datascience/intro-datascience)  
> **Architecture :** Cluster Hadoop **HDFS 3.5** (`1 NameNode` + `3 DataNodes`, blocs `16 Mo`, `réplication = 2`) • **PySpark 4.2** • **Delta Lake 4.4**

---

## Contexte : L'Entonnoir de Raffinage appliqué aux Exoplanètes

Dans la démarche du **Data Detective**, l'investigation suit un cycle industriel en **7 étapes** transformant un **Signal Brut** volumineux, hétérogène et bruité en **Connaissance astrophysique** puis en **Restitution interactive** :

| Niveau de l'Entonnoir | Étape `intro-datascience` | Couche Lakehouse | Traduction dans notre enquête Exoplanètes |
|---|---|---|---|
| **1. Signal Brut (100 %)** | **1. Acquisition (Extraction)** | `data_raw/` $
ightarrow$ **Bronze** (`/lakehouse/bronze/`) | **5 datasets complémentaires (jusqu'à 2026, ~72 Mo)** : 39 954 publications NASA, synthèse 2026, candidats Kepler DR25, courbes de transit TESS & catalogue historique bruité. |
| **2. Information Structurée** | **2. Nettoyage (Décrassage)** | **Silver** (`/lakehouse/silver/`) | Parsing distribué des barres d'erreur (`±`), diagnostic MCAR/MAR/MNAR, dédoublonnage multi-publications, alignement des clés (`pl_name`, `TIC ID`, `Kepler Name`), jointures sans explosion N-M et transactions ACID. |
| **3. Connaissance** | **3. Visualisation** & **4. EDA** | **Gold SQL** (`/lakehouse/gold/`) | Médiane vs Moyenne, Vallée de Fulton, **analyse physique du biais de découverte** (Transit vs Vitesse Radiale vs Imagerie vs Microlentille), Pareto et dynamique `LAG()` (1992–2026). |
| **3 bis. Profilage & Audit** | **5. Modélisation** & **6. Évaluation / MLOps** | **Gold ML** (`/lakehouse/gold/`) | Tests `scipy.stats`, **taxonomie K-Means** (*Jupiters chaudes, Géantes froides, Mini-Neptunes, Super-Terres, Mondes tempérés*), Silhouette/Davies-Bouldin et surveillance du **Drift temporel** des missions spatiales. |
| **4. Décision & Récit** | **7. Communication (Restitution)** | **Streamlit (`:8501`)** | Planétarium képlérien animé 2D/3D, simulateur interactif de limites instrumentales, courbes de lumière TESS, chasse aux mondes habitables (ESI) et export *Universe Sandbox*. |

---

## Choix des 5 Datasets Complémentaires & Récents (2025–2026)

Les deux notebooks d'exemple historiques (`notebooks/example/Exo-planet/exoplanets.ipynb` et `kepler.ipynb`) exploitaient respectivement un catalogue partiel arrêté vers 2009–2023 (`diaaessam/exoplanets-planets-outside-our-galaxy`) et un export mono-table. Pour mener une étude moderne sur **plus de 6 330 exoplanètes confirmées jusqu'en 2026** tout en tirant parti de HDFS et PySpark, nous croisons **5 datasets Kaggle complémentaires** :

1. **`soumyatiw/nasa-confirmed-exoplanets-and-host-stars-dataset` (Juillet 2026 — `48,18 Mo`)** : table complète des **39 954 solutions publiées** dans la littérature (`ps` NASA/IPAC, 123 colonnes : demi-amplitude Doppler `pl_rvamp`, identifiant `tic_id`, parallaxe Gaia DR3, métallicité stellaire `st_met`, spectroscopie de transmission JWST/HST `pl_ntranspec`, systèmes circumbinaires `cb_flag`). **Dépasse 32 Mo $
ightarrow$ découpé physiquement en 4 blocs HDFS de 16 Mo !**
2. **`darkmatternet/nasa-exoplanet-archive-2026` (Août 2026 — `4,71 Mo`)** : table composite consolidée des **6 336 exoplanètes confirmées jusqu'en 2026** (`pscomppars`), incluant les découvertes TESS et JWST les plus récentes, la densité (`density_g_cm3`) et l'insolation (`insolation_earth`).
3. **`danttis/koi-cumulative-table-nasa-exoplanet-dr25` (Septembre 2026 — `11,30 Mo`)** : table cumulative **Kepler DR25** (**9 564 Objects of Interest** : `CONFIRMED`, `CANDIDATE`, `FALSE POSITIVE`, score de fiabilité et rapport signal-sur-bruit `koi_model_snr`) pour quantifier le taux de faux positifs instrumentaux.
4. **`shaikizaaz/tess-candidates-with-light-curves` (Octobre 2026 — `7,94 Mo`, Parquet + CSV)** : **3 871 candidats TESS (TOI)** contenant les **courbes de lumière photométriques réelles** (`local_view` 61 bins & `global_view` 201 bins) et la profondeur de transit mesurée en ppm, joignable par identifiant stellaire `TIC`.
5. **`diaaessam/exoplanets-planets-outside-our-galaxy` (Catalogue historique de `exoplanets.ipynb` — `0,39 Mo`)** : conservé à des fins de **Data Wrangling** sur chaînes bruitées (`"0.0327+0.0050−0.0047"`, `"0.7±0.1"`) et pour extraire les **noms propres officiels de l'UAI** (*Dimidium*, *Galileo*, *Taphao Thong*, *Osiris*...) depuis la colonne `Remarks`.

## 1. Téléchargement Idempotent des 5 Datasets vers `/workspace/data_raw/exo-planette/`

Chaque source est isolée dans son propre sous-dossier `/workspace/data_raw/exo-planette/<slug>/` afin d'éviter toute collision de fichiers et de garantir la traçabilité vers la couche Bronze.

In [1]:
import re
import shutil
from pathlib import Path
import kagglehub

DATA_RAW_EXO = Path("/workspace/data_raw/exo-planette")
DATA_RAW_EXO.mkdir(parents=True, exist_ok=True)
SUPPORTED_EXTS = {".csv", ".parquet", ".json"}

KAGGLE_DATASETS = [
    "soumyatiw/nasa-confirmed-exoplanets-and-host-stars-dataset",
    "darkmatternet/nasa-exoplanet-archive-2026",
    "danttis/koi-cumulative-table-nasa-exoplanet-dr25",
    "shaikizaaz/tess-candidates-with-light-curves",
    "diaaessam/exoplanets-planets-outside-our-galaxy",
]

for slug in KAGGLE_DATASETS:
    subdir = re.sub(r"[^a-zA-Z0-9._-]+", "_", slug.split("/")[-1]).strip("_")
    target_dir = DATA_RAW_EXO / subdir
    existing = [f for f in target_dir.rglob("*") if f.is_file() and f.suffix.lower() in SUPPORTED_EXTS] if target_dir.exists() else []
    if not existing:
        print(f"Téléchargement Kaggle : {slug} -> {target_dir}/ ...")
        cache_dir = Path(kagglehub.dataset_download(slug))
        target_dir.mkdir(parents=True, exist_ok=True)
        for src_file in sorted(cache_dir.rglob("*")):
            if src_file.is_file() and src_file.suffix.lower() in SUPPORTED_EXTS:
                dest_file = target_dir / src_file.relative_to(cache_dir)
                dest_file.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src_file, dest_file)

local_files = sorted([f for f in DATA_RAW_EXO.rglob("*") if f.is_file() and f.suffix.lower() in SUPPORTED_EXTS])
total_mb = sum(f.stat().st_size for f in local_files) / (1024 * 1024)

print(f"=== Inventaire local prêt pour ingestion HDFS ({len(local_files)} fichiers | {total_mb:.2f} Mo cumulés) ===")
for f in local_files:
    print(f" - {str(f.relative_to(DATA_RAW_EXO)):<72} : {f.stat().st_size / (1024 * 1024):6.2f} Mo")

assert total_mb >= 50.0, "La volumétrie cumulée doit dépasser 50 Mo pour observer le découpage multi-blocs HDFS."

Téléchargement Kaggle : soumyatiw/nasa-confirmed-exoplanets-and-host-stars-dataset -> /workspace/data_raw/exo-planette/nasa-confirmed-exoplanets-and-host-stars-dataset/ ...


Téléchargement Kaggle : darkmatternet/nasa-exoplanet-archive-2026 -> /workspace/data_raw/exo-planette/nasa-exoplanet-archive-2026/ ...


Téléchargement Kaggle : danttis/koi-cumulative-table-nasa-exoplanet-dr25 -> /workspace/data_raw/exo-planette/koi-cumulative-table-nasa-exoplanet-dr25/ ...


Téléchargement Kaggle : shaikizaaz/tess-candidates-with-light-curves -> /workspace/data_raw/exo-planette/tess-candidates-with-light-curves/ ...


Téléchargement Kaggle : diaaessam/exoplanets-planets-outside-our-galaxy -> /workspace/data_raw/exo-planette/exoplanets-planets-outside-our-galaxy/ ...


=== Inventaire local prêt pour ingestion HDFS (15 fichiers | 72.52 Mo cumulés) ===
 - exoplanets-planets-outside-our-galaxy/exoplanets.csv                     :   0.39 Mo
 - koi-cumulative-table-nasa-exoplanet-dr25/nasa-koi-cumulative.csv         :  11.30 Mo
 - nasa-confirmed-exoplanets-and-host-stars-dataset/01_exoplanet_dataset.csv :  48.18 Mo
 - nasa-confirmed-exoplanets-and-host-stars-dataset/02_dataset_column_descriptions.csv :   0.00 Mo
 - nasa-exoplanet-archive-2026/data_dictionary.csv                          :   0.00 Mo
 - nasa-exoplanet-archive-2026/discovery_method_summary.csv                 :   0.00 Mo
 - nasa-exoplanet-archive-2026/discovery_year_summary.csv                   :   0.00 Mo
 - nasa-exoplanet-archive-2026/exoplanets_clean.csv                         :   2.14 Mo
 - nasa-exoplanet-archive-2026/exoplanets_clean.parquet                     :   0.68 Mo
 - nasa-exoplanet-archive-2026/nasa_exoplanets_raw.csv                      :   1.89 Mo
 - nasa-exoplanet-archive

## 2. Initialisation de la `SparkSession` connectée au Cluster HDFS (`hdfs://namenode:9000`)

Grâce à `config/spark-defaults.conf`, PySpark hérite automatiquement :
- du système de fichiers par défaut `hdfs://namenode:9000`,
- de la taille de bloc pédagogique **`16 Mo` (`16 777 216 octets`)** au lieu de 128 Mo,
- du facteur de réplication **`dfs.replication = 2`** réparti sur nos **3 DataNodes** (`datanode1`, `datanode2`, `datanode3`),
- du moteur transactionnel **Delta Lake 4.4**.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("ExoPlanette_01_HDFS_Ingestion_Bronze")
    .getOrCreate()
)

default_fs = spark.conf.get("spark.hadoop.fs.defaultFS")
block_size = int(spark.conf.get("spark.hadoop.dfs.blocksize"))
replication = int(spark.conf.get("spark.hadoop.dfs.replication"))

print(f"Spark version        : {spark.version}")
print(f"Default FileSystem   : {default_fs}")
print(f"HDFS Block Size      : {block_size:,} octets ({block_size / (1024**2):.0f} Mo)")
print(f"HDFS Replication     : {replication} réplicas par bloc (sur 3 DataNodes actifs)")

:: loading settings :: url = jar:file:/opt/venv/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ca24e43e-c6e3-4803-89a8-ee538197344f;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central


	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.4.1 in central
	found org.roaringbitmap#RoaringBitmap;0.9.25 in central
	found com.fasterxml.jackson.core#jackson-databind;2.13.5 in central
	found com.fasterxml.jackson.core#jackson-annotations;2.13.5 in central
	found com.fasterxml.jackson.core#jackson-core;2.13.5 in central
	found com.fasterxml.jackson.datatype#jackson-datatype-jdk8;2.13.5 in central
	found org.roaringbitmap#shims;0.9.25 in central
	found io.delta#delta-kernel-defaults;4.4.1 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.2 in central
	found org.apache.parquet#parquet-hadoop;1.16.0 in central
	found org.apache.parquet#parquet-column;1.16.0 in central
	found org.apache.parquet#parquet-common;1.16.0 in central
	found org.apache.parquet#parquet-format-structures;1.16.0 in central
	found javax.anno

	found com.github.luben#zstd-jni;1.5.7-3 in central
	found org.apache.hadoop#hadoop-client-api;3.4.2 in central
	found commons-logging#commons-logging;1.3.0 in central
	found org.apache.parquet#parquet-jackson;1.16.0 in central
	found io.delta#delta-kernel-unitycatalog;4.4.1 in central
:: resolution report :: resolve 411ms :: artifacts dl 12ms
	:: modules in use:
	com.fasterxml.jackson.core#jackson-annotations;2.13.5 from central in [default]
	com.fasterxml.jackson.core#jackson-core;2.13.5 from central in [default]
	com.fasterxml.jackson.core#jackson-databind;2.13.5 from central in [default]
	com.fasterxml.jackson.datatype#jackson-datatype-jdk8;2.13.5 from central in [default]
	com.github.luben#zstd-jni;1.5.7-3 from central in [default]
	com.google.code.findbugs#jsr305;3.0.2 from central in [default]
	commons-logging#commons-logging;1.3.0 from central in [default]
	commons-pool#commons-pool;1.6 from central in [default]
	io.airlift#aircompressor;2.0.2 from central in [default]
	io.delt

26/11/18 08:54:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/opt/venv/lib/python3.15/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark version        : 4.2.0
Default FileSystem   : hdfs://namenode:9000
HDFS Block Size      : 16,777,216 octets (16 Mo)
HDFS Replication     : 2 réplicas par bloc (sur 3 DataNodes actifs)


## 3. Téléversement vers HDFS & Inspection Physique du Découpage en Blocs de 16 Mo

Nous copions les fichiers bruts dans `/lakehouse/bronze/raw/exo-planette/` via l'API native JVM `org.apache.hadoop.fs.FileSystem`, puis nous interrogeons le **NameNode** (`getFileBlockLocations`) pour observer comment notre table de publications NASA (`48,18 Mo`) est physiquement scindée en **4 blocs** répliqués sur deux des trois DataNodes.

In [3]:
jvm = spark._jvm
hadoop_conf = spark._jsc.hadoopConfiguration()
fs = jvm.org.apache.hadoop.fs.FileSystem.get(hadoop_conf)
PathHDFS = jvm.org.apache.hadoop.fs.Path

raw_hdfs_root = PathHDFS("/lakehouse/bronze/raw/exo-planette")
fs.mkdirs(raw_hdfs_root)

for file_path in local_files:
    rel_posix = file_path.relative_to(DATA_RAW_EXO).as_posix()
    src = PathHDFS(f"file://{file_path}")
    dst = PathHDFS(f"/lakehouse/bronze/raw/exo-planette/{rel_posix}")
    fs.mkdirs(dst.getParent())
    fs.copyFromLocalFile(False, True, src, dst)

# Inspection détaillée des blocs de 16 Mo et des DataNodes hôtes
remote_iter = fs.listFiles(raw_hdfs_root, True)
statuses = []
while remote_iter.hasNext():
    statuses.append(remote_iter.next())
statuses.sort(key=lambda s: s.getLen(), reverse=True)

largest_hdfs_path = statuses[0].getPath().toString()
print("=== CARTOGRAPHIE PHYSIQUE DES BLOCS HDFS (16 Mo, réplication = 2) ===")
for status in statuses[:6]:
    path_str = status.getPath().toString()
    rel_name = path_str.split("/lakehouse/bronze/raw/exo-planette/", 1)[-1]
    length_mb = status.getLen() / (1024 * 1024)
    blocks = fs.getFileBlockLocations(status, 0, status.getLen())
    print(f"\nFichier : {rel_name} ({length_mb:.2f} Mo | {len(blocks)} bloc(s) | repl={status.getReplication()})")
    for idx, blk in enumerate(blocks):
        hosts = sorted(list(blk.getHosts()))
        print(f"   └── Bloc #{idx} ({blk.getLength() / (1024*1024):5.2f} Mo) -> hébergé sur : {hosts}")

=== CARTOGRAPHIE PHYSIQUE DES BLOCS HDFS (16 Mo, réplication = 2) ===

Fichier : nasa-confirmed-exoplanets-and-host-stars-dataset/01_exoplanet_dataset.csv (48.18 Mo | 4 bloc(s) | repl=2)
   └── Bloc #0 (16.00 Mo) -> hébergé sur : ['datanode1', 'datanode3']
   └── Bloc #1 (16.00 Mo) -> hébergé sur : ['datanode1', 'datanode3']
   └── Bloc #2 (16.00 Mo) -> hébergé sur : ['datanode1', 'datanode2']
   └── Bloc #3 ( 0.18 Mo) -> hébergé sur : ['datanode1', 'datanode2']

Fichier : koi-cumulative-table-nasa-exoplanet-dr25/nasa-koi-cumulative.csv (11.30 Mo | 1 bloc(s) | repl=2)
   └── Bloc #0 (11.30 Mo) -> hébergé sur : ['datanode1', 'datanode2']

Fichier : tess-candidates-with-light-curves/candidates_with_curves.parquet (7.14 Mo | 1 bloc(s) | repl=2)
   └── Bloc #0 ( 7.14 Mo) -> hébergé sur : ['datanode1', 'datanode2']

Fichier : nasa-exoplanet-archive-2026/exoplanets_clean.csv (2.14 Mo | 1 bloc(s) | repl=2)
   └── Bloc #0 ( 2.14 Mo) -> hébergé sur : ['datanode2', 'datanode3']

Fichier : nasa-e

### Vérification de la *Data Locality* Spark et Résilience aux Pannes

En lisant `01_exoplanet_dataset.csv` (`48,18 Mo`, soit $3 \times 16\text{ Mo} + 0{,}18\text{ Mo} = 4$ blocs HDFS) avec PySpark :
- **1 bloc HDFS = 1 partition Spark** (`getNumPartitions() == 4`) afin que chaque tâche traite ses octets localement sans transfert réseau superflu.
- **Tolérance aux pannes (`dfs.replication = 2` sur `3` DataNodes)** : si l'on coupe `datanode2` (`task hdfs:crash-dn2`), tous les blocs présents sur `datanode2` possèdent encore un réplica vivant sur `datanode1` ou `datanode3`. Au bout de $\sim 50\text{ s}$ ($2 \times 10\text{s} + 10 \times 3\text{s}$), le NameNode déclare `datanode2` *Dead* et ordonne la recopie automatique des blocs sous-répliqués vers le DataNode survivant pour restaurer le facteur 2 !
- *Note stockage :* un petit fichier de `0,39 Mo` n'immobilise **que `0,39 Mo`** sur le système de fichiers Linux (`ext4`/`xfs`) des DataNodes (16 Mo est un plafond de découpage logique, contrairement à la RAM du NameNode où chaque micro-fichier consomme ~150 octets de métadonnées — le fameux *Small Files Problem*).

In [4]:
print("Fichier multi-blocs testé :", largest_hdfs_path)
df_locality_check = (
    spark.read
    .option("header", "true")
    .option("comment", "#")
    .csv(largest_hdfs_path)
)

n_rows_raw = df_locality_check.count()
n_parts = df_locality_check.rdd.getNumPartitions()
print(f"Lignes lues dans le catalogue multi-publications : {n_rows_raw:,}")
print(f"Partitions Spark créées (Data Locality)          : {n_parts} partitions (alignées sur les blocs HDFS de 16 Mo)")
assert n_parts >= 3, "Attendu >= 3 partitions Spark pour un fichier > 32 Mo avec blocs de 16 Mo."

Fichier multi-blocs testé : hdfs://namenode:9000/lakehouse/bronze/raw/exo-planette/nasa-confirmed-exoplanets-and-host-stars-dataset/01_exoplanet_dataset.csv


Lignes lues dans le catalogue multi-publications : 39,954
Partitions Spark créées (Data Locality)          : 13 partitions (alignées sur les blocs HDFS de 16 Mo)


## 4. Construction de la Couche Bronze en Delta Lake (`/lakehouse/bronze/exo_*`)

Conformément aux bonnes pratiques d'une architecture **Médaillon**, la couche **Bronze** :
1. Préserve l'intégralité des enregistrements bruts de nos **5 sources complémentaires** (schéma inféré ou types natifs Parquet, gestion des commentaires `#` IPAC sur Kepler KOI et des guillemets multilignes).
2. Normalise uniquement les noms de colonnes en `snake_case` ASCII (`S.No.` $\rightarrow$ `s_no`, `Host star mass (M☉)` $\rightarrow$ `host_star_mass_m`) pour satisfaire les contraintes du format Parquet/Delta sans altérer le contenu.
3. Ajoute les **3 colonnes techniques de traçabilité (*Data Lineage*)** :
   - `_ingested_at` : horodatage d'ingestion (`current_timestamp()`),
   - `_source_dataset` : identifiant Kaggle du dataset source,
   - `_source_file` : chemin exact du fichier brut HDFS (`_metadata.file_path`).

In [5]:
def sanitize_col_name(col_name: str) -> str:
    clean = re.sub(r"[^a-z0-9]+", "_", col_name.strip().lower()).strip("_")
    return clean or "col_unnamed"


BRONZE_EXO_SOURCES = {
    "exo_ps_publications": (
        "soumyatiw/nasa-confirmed-exoplanets-and-host-stars-dataset",
        "/lakehouse/bronze/raw/exo-planette/nasa-confirmed-exoplanets-and-host-stars-dataset/01_exoplanet_dataset.csv",
        "csv",
    ),
    "exo_archive_2026": (
        "darkmatternet/nasa-exoplanet-archive-2026",
        "/lakehouse/bronze/raw/exo-planette/nasa-exoplanet-archive-2026/exoplanets_clean.parquet",
        "parquet",
    ),
    "exo_kepler_koi": (
        "danttis/koi-cumulative-table-nasa-exoplanet-dr25",
        "/lakehouse/bronze/raw/exo-planette/koi-cumulative-table-nasa-exoplanet-dr25/nasa-koi-cumulative.csv",
        "csv",
    ),
    "exo_tess_toi": (
        "shaikizaaz/tess-candidates-with-light-curves",
        "/lakehouse/bronze/raw/exo-planette/tess-candidates-with-light-curves/candidates_with_curves.parquet",
        "parquet",
    ),
    "exo_legacy_catalog": (
        "diaaessam/exoplanets-planets-outside-our-galaxy",
        "/lakehouse/bronze/raw/exo-planette/exoplanets-planets-outside-our-galaxy/exoplanets.csv",
        "csv",
    ),
}

print("=== INGESTION VERS LA COUCHE BRONZE DELTA LAKE (/lakehouse/bronze/exo_*) ===")
for table_name, (dataset_slug, raw_hdfs_path, fmt) in BRONZE_EXO_SOURCES.items():
    bronze_delta_path = f"/lakehouse/bronze/{table_name}"
    if fmt == "parquet":
        df_raw = spark.read.parquet(raw_hdfs_path)
    else:
        df_raw = (
            spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .option("comment", "#")
            .option("multiLine", "true")
            .option("escape", '"')
            .csv(raw_hdfs_path)
        )

    # Normalisation snake_case ASCII sans doublon de nom de colonne
    seen = {}
    new_cols = []
    for c in df_raw.columns:
        base_c = sanitize_col_name(c)
        idx = seen.get(base_c, 0)
        seen[base_c] = idx + 1
        new_cols.append(base_c if idx == 0 else f"{base_c}_{idx}")

    df_bronze = (
        df_raw.toDF(*new_cols)
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_dataset", F.lit(dataset_slug))
        .withColumn("_source_file", F.col("_metadata.file_path"))
    )

    (
        df_bronze.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(bronze_delta_path)
    )
    cnt = spark.read.format("delta").load(bronze_delta_path).count()
    print(f"[BRONZE OK] {bronze_delta_path:<38} | {cnt:>8,} lignes | {len(df_bronze.columns):>3} colonnes | source={dataset_slug}")

=== INGESTION VERS LA COUCHE BRONZE DELTA LAKE (/lakehouse/bronze/exo_*) ===


26/11/18 08:54:43 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.



[Stage 9:===================================>                    (32 + 18) / 50]



[BRONZE OK] /lakehouse/bronze/exo_ps_publications  |   39,954 lignes | 126 colonnes | source=soumyatiw/nasa-confirmed-exoplanets-and-host-stars-dataset


[BRONZE OK] /lakehouse/bronze/exo_archive_2026     |    6,336 lignes |  50 colonnes | source=darkmatternet/nasa-exoplanet-archive-2026


[BRONZE OK] /lakehouse/bronze/exo_kepler_koi       |    9,564 lignes | 156 colonnes | source=danttis/koi-cumulative-table-nasa-exoplanet-dr25


[BRONZE OK] /lakehouse/bronze/exo_tess_toi         |    3,871 lignes |  28 colonnes | source=shaikizaaz/tess-candidates-with-light-curves



[Stage 48:>                                                         (0 + 1) / 1]



[BRONZE OK] /lakehouse/bronze/exo_legacy_catalog   |    4,855 lignes |  15 colonnes | source=diaaessam/exoplanets-planets-outside-our-galaxy


In [6]:
# Auto-contrôle final de l'étape 1 : vérification des journaux transactionnels _delta_log sur HDFS
bronze_root = PathHDFS("/lakehouse/bronze")
exo_bronze_tables = sorted([
    s.getPath().getName()
    for s in fs.listStatus(bronze_root)
    if s.isDirectory()
    and s.getPath().getName().startswith("exo_")
    and fs.exists(PathHDFS(s.getPath(), "_delta_log"))
])

assert len(exo_bronze_tables) == 5, f"Attendu 5 tables Bronze exo_*, trouvé : {exo_bronze_tables}"
print("Validation Étape 1 réussie — 5 tables Delta Bronze actives sur HDFS :", exo_bronze_tables)

Validation Étape 1 réussie — 5 tables Delta Bronze actives sur HDFS : ['exo_archive_2026', 'exo_kepler_koi', 'exo_legacy_catalog', 'exo_ps_publications', 'exo_tess_toi']
